# חיפוש תמונות בעברית: השוואת מודלים
**שלב 1: הכנת המאגר, יצירת וקטורים, מדידה והעלאה ל-Hugging Face**

לפני ההרצה: ב-Colab בחר `Runtime → Change runtime type → T4 GPU`. ב-Kaggle: `Settings → Accelerator → GPU`.

זמן ריצה משוער: 15–25 דקות (רוב הזמן הוא התרגום).

In [ ]:
!pip install -q sentence-transformers datasets transformers sentencepiece sacremoses huggingface_hub pandas

In [ ]:
import os, re, json, random
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import MarianMTModel, MarianTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU found - this will be very slow. Switch the runtime to GPU.")

OUT = "hebrew_clip_search"
os.makedirs(f"{OUT}/images", exist_ok=True)
os.makedirs(f"{OUT}/embeddings", exist_ok=True)
random.seed(42)

## 1. טעינת Flickr30k (סט הבדיקה של 1,000 תמונות)
הדפסת המבנה עוזרת לוודא שהעמודות הן מה שאנחנו מצפים להן.

In [ ]:
# The repo uses an old loading script that new `datasets` versions reject,
# so we read Hugging Face's auto-converted Parquet copy directly.
import io
from PIL import Image

PARQUET = "hf://datasets/nlphuji/flickr_1k_test_image_text_retrieval@refs%2Fconvert%2Fparquet/**/*.parquet"
ds = load_dataset("parquet", data_files=PARQUET, split="train")
print(ds)

def to_pil(x):
    # usually already a PIL image; fall back to raw bytes if not
    if isinstance(x, dict):
        x = Image.open(io.BytesIO(x["bytes"]))
    return x

def as_list(c):
    # captions may arrive as a list or as a JSON-encoded string
    if isinstance(c, str):
        try:
            c = json.loads(c)
        except json.JSONDecodeError:
            c = [c]
    return list(c)

images, captions_en, caption_img_idx = [], [], []
for i, ex in enumerate(tqdm(ds, desc="Loading")):
    img = to_pil(ex["image"]).convert("RGB")
    images.append(img)
    img.save(f"{OUT}/images/{i:04d}.jpg", quality=90)
    for cap in as_list(ex["caption"]):
        captions_en.append(cap)
        caption_img_idx.append(i)

caption_img_idx = np.array(caption_img_idx)
print(f"{len(images)} images, {len(captions_en)} captions")
print("Example:", captions_en[0])
images[0]

## 2. תרגום הכיתובים לעברית, ותרגום חזרה לאנגלית
- `en → he`: יוצר את שאילתות הבדיקה בעברית.
- `he → en`: מה שהגישה של "תרגום ואז CLIP" תקבל בזמן אמת.

In [ ]:
def translate(texts, model_name, bs=64):
    tok = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name).to(device).eval()
    if device == "cuda":
        model = model.half()
    out = []
    for i in tqdm(range(0, len(texts), bs), desc=model_name.split("/")[-1]):
        batch = tok(texts[i:i+bs], return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            gen = model.generate(**batch, max_length=256, num_beams=4)
        out += tok.batch_decode(gen, skip_special_tokens=True)
    del model
    torch.cuda.empty_cache()
    return out

captions_he = translate(captions_en, "Helsinki-NLP/opus-mt-en-he")
captions_he2en = translate(captions_he, "Helsinki-NLP/opus-mt-tc-big-he-en")

for i in random.sample(range(len(captions_en)), 5):
    print("EN   :", captions_en[i])
    print("HE   :", captions_he[i])
    print("HE>EN:", captions_he2en[i])
    print("-" * 60)

In [ ]:
# Save all captions, plus a 100-row sample for manual review
captions = [
    {"img": int(caption_img_idx[i]), "en": captions_en[i], "he": captions_he[i], "he2en": captions_he2en[i]}
    for i in range(len(captions_en))
]
with open(f"{OUT}/captions.json", "w", encoding="utf-8") as f:
    json.dump(captions, f, ensure_ascii=False, indent=1)

review_idx = sorted(random.sample(range(len(captions)), 100))
pd.DataFrame(
    [{"row": i, "img": captions[i]["img"], "en": captions[i]["en"], "he_machine": captions[i]["he"], "he_fixed": ""} for i in review_idx]
).to_csv(f"{OUT}/manual_review.csv", index=False, encoding="utf-8-sig")
print("Saved captions.json and manual_review.csv")

## 3. וקטורים לתמונות: שתי ריצות בלבד
- **ViT-B/32**: משותף ל-CLIP באנגלית ולמודל הרב-לשוני.
- **ViT-L/14**: משותף ל-CLIP הגדול באנגלית ול-HebCLIP.

In [ ]:
IMAGE_ENCODERS = {"B32": "clip-ViT-B-32", "L14": "clip-ViT-L-14"}
img_emb = {}

for key, name in IMAGE_ENCODERS.items():
    model = SentenceTransformer(name, device=device)
    emb = model.encode(images, batch_size=64, convert_to_numpy=True,
                       normalize_embeddings=True, show_progress_bar=True)
    img_emb[key] = emb.astype(np.float32)
    np.save(f"{OUT}/embeddings/images_{key}.npy", img_emb[key])
    print(key, emb.shape)
    del model
    torch.cuda.empty_cache()

## 4. מדידה: Recall@K
לכל שאילתה בודקים אם התמונה הנכונה הופיעה ב-K התוצאות הראשונות.

שתי השורות עם הכיתובים המקוריים באנגלית הן **תקרת ייחוס**: כמה טוב המודל עובד בלי שום עברית. הפער מהן מראה כמה "עולה" לנו העברית.

In [ ]:
def recall_at_k(q, imgs, gt, ks=(1, 5, 10)):
    sims = torch.from_numpy(q) @ torch.from_numpy(imgs).T
    top = sims.topk(max(ks), dim=1).indices.numpy()
    hits = top == gt[:, None]
    return {f"R@{k}": round(hits[:, :k].any(axis=1).mean() * 100, 2) for k in ks}

def encode_text(model_name, texts):
    model = SentenceTransformer(model_name, device=device)
    emb = model.encode(texts, batch_size=256, convert_to_numpy=True,
                       normalize_embeddings=True, show_progress_bar=True)
    del model
    torch.cuda.empty_cache()
    return emb.astype(np.float32)

# (row name, text model, input texts, image encoder)
EXPERIMENTS = [
    ("Reference: English captions, CLIP B/32",       "clip-ViT-B-32",                                     captions_en,    "B32"),
    ("Hebrew -> translate -> CLIP B/32",            "clip-ViT-B-32",                                     captions_he2en, "B32"),
    ("Hebrew direct, multilingual CLIP B/32",       "sentence-transformers/clip-ViT-B-32-multilingual-v1", captions_he,   "B32"),
    ("Reference: English captions, CLIP L/14",       "clip-ViT-L-14",                                     captions_en,    "L14"),
    ("Hebrew -> translate -> CLIP L/14",            "clip-ViT-L-14",                                     captions_he2en, "L14"),
]

results = []
for row_name, model_name, texts, enc in EXPERIMENTS:
    print("\n==>", row_name)
    q = encode_text(model_name, texts)
    tag = re.sub(r"[^A-Za-z0-9]+", "_", row_name).strip("_")  # safe file name (no "/")
    np.save(f"{OUT}/embeddings/text_{enc}_{tag}.npy", q)
    r = recall_at_k(q, img_emb[enc], caption_img_idx)
    results.append({"method": row_name, "image_encoder": enc, **r})
    print(r)

df = pd.DataFrame(results)
df.to_csv(f"{OUT}/results.csv", index=False)
df

## 5. HebCLIP של NVIDIA
לא הצלחתי לאמת מראש את קוד הטעינה המדויק שלו, אז במקום לנחש, התא הזה מדפיס את דף המודל.
**העתק לי את הפלט**, ואכתוב את שורת ה-HebCLIP לטבלה (היא תשתמש בווקטורי `L14` שכבר חישבנו).

אם יש שגיאת הרשאה: היכנס לדף המודל ב-Hugging Face, אשר את תנאי הרישיון, והרץ קודם את תא ההתחברות בשלב 6.

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
print(list_repo_files("nvidia/hebclip-model"))
readme = hf_hub_download("nvidia/hebclip-model", "README.md")
print(open(readme, encoding="utf-8").read())

## 6. העלאה ל-Hugging Face
1. צור טוקן ב-https://huggingface.co/settings/tokens עם הרשאת **Write**.
2. שנה את `HF_USERNAME` לשם המשתמש שלך.

התוצאות עולות כ-**Dataset** (לא Space). ה-Space שנבנה אחר כך יוריד משם את הווקטורים והתמונות.

In [ ]:
from huggingface_hub import login, HfApi
login()  # paste your Write token when prompted

HF_USERNAME = "your-username"   # <-- change this
REPO_ID = f"{HF_USERNAME}/hebrew-clip-search-data"

api = HfApi()
api.create_repo(REPO_ID, repo_type="dataset", exist_ok=True, private=False)
api.upload_folder(folder_path=OUT, repo_id=REPO_ID, repo_type="dataset",
                  commit_message="Flickr30k 1k test: Hebrew captions, embeddings, results")
print(f"Done: https://huggingface.co/datasets/{REPO_ID}")